In [0]:
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.bronze_transactions")
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.silver_transactions")
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.gold_sar_reports")
dbutils.fs.rm("/Volumes/aml_pipeline/transactions/raw_data/_checkpoints", recurse=True)

tables = spark.sql("SHOW TABLES IN aml_pipeline.transactions").collect()
print(f"Remaining tables: {len(tables)}")
for t in tables:
    print(f"  {t.tableName}")
print("All clear")


Remaining tables: 0
All clear


In [0]:
import requests
from pyspark.sql.functions import col, udf, when, lit, current_timestamp, concat, date_format
from pyspark.sql.types import DoubleType, StringType, StructType, StructField, IntegerType

RAW_PATH        = "/Volumes/aml_pipeline/transactions/raw_data/"
BRONZE_TABLE    = "aml_pipeline.transactions.bronze_transactions"
SILVER_TABLE    = "aml_pipeline.transactions.silver_transactions"
GOLD_TABLE      = "aml_pipeline.transactions.gold_sar_reports"
CHECKPOINT_PATH = "/Volumes/aml_pipeline/transactions/raw_data/_checkpoints/bronze_v4"

# STEP 1: Bronze
print("STEP 1: Bronze ingestion...")

schema = StructType([
    StructField("transaction_id",   StringType(),  True),
    StructField("timestamp",        StringType(),  True),
    StructField("message_type",     StringType(),  True),
    StructField("batch_number",     IntegerType(), True),
    StructField("sender_name",      StringType(),  True),
    StructField("sender_account",   StringType(),  True),
    StructField("sender_country",   StringType(),  True),
    StructField("sender_address",   StringType(),  True),
    StructField("receiver_name",    StringType(),  True),
    StructField("receiver_account", StringType(),  True),
    StructField("receiver_country", StringType(),  True),
    StructField("amount",           DoubleType(),  True),
    StructField("currency",         StringType(),  True),
    StructField("purpose_code",     StringType(),  True),
    StructField("transaction_type", StringType(),  True),
])

bronze_query = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
         .schema(schema)
         .load(RAW_PATH)
         .withColumn("ingestion_timestamp", current_timestamp())
         .withColumn("source_file",         col("_metadata.file_path"))
         .withColumn("pipeline_layer",      lit("bronze"))
         .writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", CHECKPOINT_PATH)
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(BRONZE_TABLE)
)
bronze_query.awaitTermination()

bronze_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze complete: {bronze_count:,} records")
assert bronze_count == 100_000, f"ERROR: Expected 100,000 but got {bronze_count:,}. Stop and fix before continuing."

# STEP 2: Silver
print("\nSTEP 2: Silver enrichment...")

try:
    rates = requests.get("https://api.frankfurter.app/latest?from=USD", timeout=10).json().get("rates", {})
    rates["USD"] = 1.0
    print(f"Live exchange rates fetched ({len(rates)} currencies)")
except:
    rates = {"USD":1.0,"EUR":0.92,"GBP":0.79,"JPY":149.5,"CHF":0.89,"CAD":1.36,"AUD":1.53,"SGD":1.34}
    print("Using fallback exchange rates")

try:
    lines = requests.get("https://www.treasury.gov/ofac/downloads/sdn.csv", timeout=15).text.split("\n")
    OFAC  = set()
    for line in lines[:500]:
        parts = line.split(",")
        if len(parts) > 1:
            n = parts[1].strip().strip('"').lower()
            if n:
                OFAC.add(n)
    print(f"OFAC list loaded ({len(OFAC)} sanctioned names)")
except:
    OFAC = {"viktor bout","semion mogilevich","ramzan kadyrov","ali khamenei","kim jong un"}
    print("Using fallback OFAC list")

HIGH_RISK           = {"KP","IR","MM","RU","BY","CU","SY","YE"}
LARGE_TXN_THRESHOLD = 500_000

def to_usd(amount, currency):
    if not amount or not currency:
        return None
    return round(float(amount) / float(rates.get(currency, 1.0)), 2)

def check_ofac(name):
    if not name:
        return "CLEAN"
    n = name.lower().strip()
    if n in OFAC:
        return "SANCTIONS_HIT"
    for s in OFAC:
        if s and len(s) > 5 and (s in n or n in s):
            return "SANCTIONS_HIT"
    return "CLEAN"

def check_travel_rule(address, sender, s_acct, receiver, r_acct):
    missing = [f for f, v in [
        ("sender_address",   address),
        ("sender_name",      sender),
        ("sender_account",   s_acct),
        ("receiver_name",    receiver),
        ("receiver_account", r_acct)
    ] if not v or not v.strip()]
    return f"TRAVEL_RULE_VIOLATION: missing {', '.join(missing)}" if missing else "COMPLIANT"

udf_usd  = udf(to_usd, DoubleType())
udf_ofac = udf(check_ofac, StringType())
udf_tr   = udf(check_travel_rule, StringType())

silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("amount_usd",                udf_usd(col("amount"), col("currency")))
    .withColumn("sender_sanctions_status",   udf_ofac(col("sender_name")))
    .withColumn("receiver_sanctions_status", udf_ofac(col("receiver_name")))
    .withColumn("travel_rule_status",        udf_tr(col("sender_address"), col("sender_name"),
                                             col("sender_account"), col("receiver_name"),
                                             col("receiver_account")))
    .withColumn("high_risk_country",         col("sender_country").isin(list(HIGH_RISK)))
    .withColumn("large_transaction",         col("amount_usd") > LARGE_TXN_THRESHOLD)
    .withColumn("is_flagged",
        (col("sender_sanctions_status") == "SANCTIONS_HIT") |
        (col("receiver_sanctions_status") == "SANCTIONS_HIT") |
        (col("travel_rule_status") != "COMPLIANT") |
        (col("high_risk_country") == True) |
        (col("large_transaction") == True))
    .withColumn("flag_reason",
        when(col("sender_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned sender"))
        .when(col("receiver_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned receiver"))
        .when(col("travel_rule_status") != "COMPLIANT", col("travel_rule_status"))
        .when(col("high_risk_country") == True, lit("HIGH_RISK_COUNTRY"))
        .when(col("large_transaction") == True, lit("LARGE_TRANSACTION_>500K_USD"))
        .otherwise(lit("NONE")))
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("pipeline_layer",   lit("silver"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

silver_count  = spark.table(SILVER_TABLE).count()
flagged_count = spark.table(SILVER_TABLE).filter("is_flagged = true").count()
clean_count   = silver_count - flagged_count
print(f"Silver complete: {silver_count:,} records")
print(f"  Flagged : {flagged_count:,} ({round(flagged_count/silver_count*100,1)}%)")
print(f"  Clean   : {clean_count:,} ({round(clean_count/silver_count*100,1)}%)")

# STEP 3: Gold
print("\nSTEP 3: Gold SAR reports...")

gold_df = (
    spark.table(SILVER_TABLE)
    .filter(col("is_flagged") == True)
    .withColumn("sar_severity",
        when((col("sender_sanctions_status") == "SANCTIONS_HIT") |
             (col("receiver_sanctions_status") == "SANCTIONS_HIT"), lit("CRITICAL"))
        .when(col("high_risk_country") == True, lit("HIGH"))
        .when(col("travel_rule_status") != "COMPLIANT", lit("HIGH"))
        .otherwise(lit("MEDIUM")))
    .withColumn("sar_reference",
        concat(lit("SAR-"), date_format(current_timestamp(), "yyyyMMdd"),
               lit("-"), col("transaction_id").substr(1, 8)))
    .withColumn("report_status",         lit("PENDING_REVIEW"))
    .withColumn("reporting_institution", lit("SentinelFlow Demo Bank"))
    .withColumn("filing_deadline",       date_format(current_timestamp(), "yyyy-MM-dd"))
    .withColumn("gold_timestamp",        current_timestamp())
    .withColumn("pipeline_layer",        lit("gold"))
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)

gold_count = spark.table(GOLD_TABLE).count()
critical   = spark.table(GOLD_TABLE).filter("sar_severity = 'CRITICAL'").count()
high       = spark.table(GOLD_TABLE).filter("sar_severity = 'HIGH'").count()
medium     = spark.table(GOLD_TABLE).filter("sar_severity = 'MEDIUM'").count()

print(f"\n{'=' * 55}")
print(f"  SENTINELFLOW PIPELINE COMPLETE")
print(f"{'=' * 55}")
print(f"  Bronze    : {bronze_count:,} raw transactions")
print(f"  Silver    : {silver_count:,} screened transactions")
print(f"  Gold      : {gold_count:,} SAR reports")
print(f"  Flag rate : {round(gold_count/silver_count*100,1)}%")
print(f"\n  SAR Severity:")
print(f"  CRITICAL  : {critical:,}")
print(f"  HIGH      : {high:,}")
print(f"  MEDIUM    : {medium:,}")
print(f"\n  Flag reason breakdown:")
spark.table(GOLD_TABLE).groupBy("flag_reason").count().orderBy("count", ascending=False).show(truncate=False)

# ── STEP 1: Bronze ───────────────────────────────────────────
print("=" * 55)
print("STEP 1: Bronze — ingesting 100,000 transactions...")
print("=" * 55)

schema = StructType([
    StructField("transaction_id",   StringType(),  True),
    StructField("timestamp",        StringType(),  True),
    StructField("message_type",     StringType(),  True),
    StructField("batch_number",     IntegerType(), True),
    StructField("sender_name",      StringType(),  True),
    StructField("sender_account",   StringType(),  True),
    StructField("sender_country",   StringType(),  True),
    StructField("sender_address",   StringType(),  True),
    StructField("receiver_name",    StringType(),  True),
    StructField("receiver_account", StringType(),  True),
    StructField("receiver_country", StringType(),  True),
    StructField("amount",           DoubleType(),  True),
    StructField("currency",         StringType(),  True),
    StructField("purpose_code",     StringType(),  True),
    StructField("transaction_type", StringType(),  True),
])

bronze_query = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
         .schema(schema)
         .load(RAW_PATH)
         .withColumn("ingestion_timestamp", current_timestamp())
         .withColumn("source_file",         col("_metadata.file_path"))
         .withColumn("pipeline_layer",      lit("bronze"))
         .writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", CHECKPOINT_PATH)
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(BRONZE_TABLE)
)
bronze_query.awaitTermination()

bronze_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze complete: {bronze_count:,} records")

# ── STEP 2: Silver ───────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 2: Silver — enrichment + compliance screening...")
print("=" * 55)

# Fetch live exchange rates
try:
    rates = requests.get("https://api.frankfurter.app/latest?from=USD", timeout=10).json().get("rates", {})
    rates["USD"] = 1.0
    print(f"Live exchange rates fetched ({len(rates)} currencies)")
except:
    rates = {"USD":1.0,"EUR":0.92,"GBP":0.79,"JPY":149.5,"CHF":0.89,"CAD":1.36,"AUD":1.53,"SGD":1.34}
    print("Using fallback exchange rates")

# Fetch OFAC sanctions list
try:
    lines = requests.get("https://www.treasury.gov/ofac/downloads/sdn.csv", timeout=15).text.split("\n")
    OFAC  = set()
    for line in lines[:500]:
        parts = line.split(",")
        if len(parts) > 1:
            n = parts[1].strip().strip('"').lower()
            if n:
                OFAC.add(n)
    print(f"OFAC list loaded ({len(OFAC)} sanctioned names)")
except:
    OFAC = {"viktor bout","semion mogilevich","ramzan kadyrov","ali khamenei","kim jong un"}
    print("Using fallback OFAC list")

HIGH_RISK = {"KP","IR","MM","RU","BY","CU","SY","YE"}

# ── Key fix: threshold raised to $500,000 ───────────────────
# Real banks flag transactions over $500k as large.
# $50k was catching normal business payments — too noisy.
LARGE_TXN_THRESHOLD = 500_000

def to_usd(amount, currency):
    if not amount or not currency:
        return None
    return round(float(amount) / float(rates.get(currency, 1.0)), 2)

def check_ofac(name):
    if not name:
        return "CLEAN"
    n = name.lower().strip()
    if n in OFAC:
        return "SANCTIONS_HIT"
    for s in OFAC:
        if s and len(s) > 5 and (s in n or n in s):
            return "SANCTIONS_HIT"
    return "CLEAN"

def check_travel_rule(address, sender, s_acct, receiver, r_acct):
    missing = [f for f, v in [
        ("sender_address",  address),
        ("sender_name",     sender),
        ("sender_account",  s_acct),
        ("receiver_name",   receiver),
        ("receiver_account",r_acct)
    ] if not v or not v.strip()]
    return f"TRAVEL_RULE_VIOLATION: missing {', '.join(missing)}" if missing else "COMPLIANT"

udf_usd  = udf(to_usd, DoubleType())
udf_ofac = udf(check_ofac, StringType())
udf_tr   = udf(check_travel_rule, StringType())

silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("amount_usd",                udf_usd(col("amount"), col("currency")))
    .withColumn("sender_sanctions_status",   udf_ofac(col("sender_name")))
    .withColumn("receiver_sanctions_status", udf_ofac(col("receiver_name")))
    .withColumn("travel_rule_status",        udf_tr(col("sender_address"), col("sender_name"),
                                             col("sender_account"), col("receiver_name"),
                                             col("receiver_account")))
    .withColumn("high_risk_country",         col("sender_country").isin(list(HIGH_RISK)))
    .withColumn("large_transaction",         col("amount_usd") > LARGE_TXN_THRESHOLD)
    .withColumn("is_flagged",
        (col("sender_sanctions_status") == "SANCTIONS_HIT") |
        (col("receiver_sanctions_status") == "SANCTIONS_HIT") |
        (col("travel_rule_status") != "COMPLIANT") |
        (col("high_risk_country") == True) |
        (col("large_transaction") == True))
    .withColumn("flag_reason",
        when(col("sender_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned sender"))
        .when(col("receiver_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned receiver"))
        .when(col("travel_rule_status") != "COMPLIANT", col("travel_rule_status"))
        .when(col("high_risk_country") == True, lit("HIGH_RISK_COUNTRY"))
        .when(col("large_transaction") == True, lit("LARGE_TRANSACTION_>500K_USD"))
        .otherwise(lit("NONE")))
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("pipeline_layer",   lit("silver"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

silver_count  = spark.table(SILVER_TABLE).count()
flagged_count = spark.table(SILVER_TABLE).filter("is_flagged = true").count()
clean_count   = silver_count - flagged_count
print(f"Silver complete: {silver_count:,} records")
print(f"   Flagged : {flagged_count:,} ({round(flagged_count/silver_count*100,1)}%)")
print(f"   Clean   : {clean_count:,} ({round(clean_count/silver_count*100,1)}%)")

# ── STEP 3: Gold ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 3: Gold — generating SAR reports...")
print("=" * 55)

gold_df = (
    spark.table(SILVER_TABLE)
    .filter(col("is_flagged") == True)
    .withColumn("sar_severity",
        when((col("sender_sanctions_status") == "SANCTIONS_HIT") |
             (col("receiver_sanctions_status") == "SANCTIONS_HIT"), lit("CRITICAL"))
        .when(col("high_risk_country") == True, lit("HIGH"))
        .when(col("travel_rule_status") != "COMPLIANT", lit("HIGH"))
        .otherwise(lit("MEDIUM")))
    .withColumn("sar_reference",
        concat(lit("SAR-"), date_format(current_timestamp(), "yyyyMMdd"),
               lit("-"), col("transaction_id").substr(1, 8)))
    .withColumn("report_status",         lit("PENDING_REVIEW"))
    .withColumn("reporting_institution", lit("SentinelFlow Demo Bank"))
    .withColumn("filing_deadline",       date_format(current_timestamp(), "yyyy-MM-dd"))
    .withColumn("gold_timestamp",        current_timestamp())
    .withColumn("pipeline_layer",        lit("gold"))
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)

gold_count    = spark.table(GOLD_TABLE).count()
critical      = spark.table(GOLD_TABLE).filter("sar_severity = 'CRITICAL'").count()
high          = spark.table(GOLD_TABLE).filter("sar_severity = 'HIGH'").count()
medium        = spark.table(GOLD_TABLE).filter("sar_severity = 'MEDIUM'").count()

# ── FINAL SUMMARY ─────────────────────────────────────────────
print(f"\n{'=' * 55}")
print(f"  SENTINELFLOW — PIPELINE COMPLETE")
print(f"{'=' * 55}")
print(f"  🥉 Bronze : {bronze_count:,} raw transactions")
print(f"  🥈 Silver : {silver_count:,} screened transactions")
print(f"  🥇 Gold   : {gold_count:,} SAR reports")
print(f"  Flag rate : {round(gold_count/silver_count*100,1)}%")
print(f"\n  SAR Severity:")
print(f"  🔴 CRITICAL : {critical:,}")
print(f"  🟠 HIGH     : {high:,}")
print(f"  🟡 MEDIUM   : {medium:,}")
print(f"\n  Flag reason breakdown:")
spark.table(GOLD_TABLE).groupBy("flag_reason").count().orderBy("count", ascending=False).show(truncate=False)

STEP 1: Bronze ingestion...
Bronze complete: 100,000 records

STEP 2: Silver enrichment...
Live exchange rates fetched (30 currencies)
OFAC list loaded (470 sanctioned names)
Silver complete: 100,000 records
  Flagged : 12,990 (13.0%)
  Clean   : 87,010 (87.0%)

STEP 3: Gold SAR reports...

  SENTINELFLOW PIPELINE COMPLETE
  Bronze    : 100,000 raw transactions
  Silver    : 100,000 screened transactions
  Gold      : 12,990 SAR reports
  Flag rate : 13.0%

  SAR Severity:
  CRITICAL  : 10,004
  HIGH      : 2,986
  MEDIUM    : 0

  Flag reason breakdown:
+---------------------------------------------+-----+
|flag_reason                                  |count|
+---------------------------------------------+-----+
|OFAC: Sanctioned sender                      |5111 |
|OFAC: Sanctioned receiver                    |4893 |
|TRAVEL_RULE_VIOLATION: missing sender_address|1500 |
|HIGH_RISK_COUNTRY                            |1486 |
+---------------------------------------------+-----+

STEP 1

In [0]:
import json
import random
import uuid
from datetime import datetime, timezone, timedelta

SANCTIONED_NAMES = [
    "Viktor Bout", "Semion Mogilevich", "Joaquin Guzman Loera",
    "Alisher Usmanov", "Ramzan Kadyrov", "Ali Khamenei",
    "Kim Jong Un", "Robert Mugabe", "Gennady Timchenko"
]
HIGH_RISK_COUNTRIES = ["KP", "IR", "MM", "RU", "BY", "CU", "SY", "YE"]
NORMAL_COUNTRIES    = ["US", "GB", "DE", "FR", "JP", "CA", "AU", "SG", "NL", "CH", "IN", "BR"]
CURRENCIES          = ["USD", "EUR", "GBP", "JPY", "CHF", "CAD", "AUD", "SGD"]
PURPOSE_CODES       = ["SALA", "SUPP", "TRAD", "LOAN", "INVS", "GDDS", "SVCS"]
NORMAL_NAMES        = [
    "Alice Johnson", "James Smith", "Maria Santos", "Wei Zhang",
    "Priya Patel", "Carlos Rivera", "Emma Wilson", "Liam Brown",
    "Yuki Tanaka", "Fatima Al-Hassan", "David Okonkwo", "Sophie Mueller",
    "Raj Sharma", "Ana Oliveira", "Chen Wei", "Mohammed Al-Rashid",
    "Isabella Ferrari", "Hiroshi Nakamura", "Amara Diallo", "Lucas Petit"
]

def make_transaction(i, batch_num, time_offset_hours=0):
    is_suspicious = (i % 20 == 0)
    txn_type = random.choice(["sanctioned", "missing_fields", "large_amount"]) if is_suspicious else "normal"
    base_time = datetime(2026, 5, 10, 8, 0, 0, tzinfo=timezone.utc)
    txn_time  = base_time + timedelta(
        hours=time_offset_hours,
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    return {
        "transaction_id":    str(uuid.uuid4()),
        "timestamp":         txn_time.isoformat(),
        "message_type":      "pacs.008",
        "batch_number":      batch_num,
        "sender_name":       random.choice(SANCTIONED_NAMES) if txn_type == "sanctioned" else random.choice(NORMAL_NAMES),
        "sender_account":    f"DE{''.join([str(random.randint(0,9)) for _ in range(18)])}",
        "sender_country":    random.choice(HIGH_RISK_COUNTRIES) if txn_type == "large_amount" else random.choice(NORMAL_COUNTRIES),
        "sender_address":    "" if txn_type == "missing_fields" else f"{random.randint(1,999)} Main St, City",
        "receiver_name":     random.choice(NORMAL_NAMES),
        "receiver_account":  f"GB{''.join([str(random.randint(0,9)) for _ in range(18)])}",
        "receiver_country":  random.choice(NORMAL_COUNTRIES),
        # KEY FIX: normal transactions capped at $10,000
        # large_amount transactions stay between $500k - $5M
        "amount":            round(random.uniform(500_000, 5_000_000), 2) if txn_type == "large_amount" else round(random.uniform(50, 10_000), 2),
        "currency":          random.choice(CURRENCIES),
        "purpose_code":      random.choice(PURPOSE_CODES),
        "transaction_type":  txn_type,
    }

# Delete old batch files and regenerate
VOLUME_PATH = "/Volumes/aml_pipeline/transactions/raw_data"
BATCH_SIZE  = 10_000
NUM_BATCHES = 10

# Delete existing batch files
print("Deleting old batch files...")
for batch_num in range(2, NUM_BATCHES + 2):
    file_path = f"{VOLUME_PATH}/batch_{str(batch_num).zfill(3)}.json"
    try:
        dbutils.fs.rm(file_path)
    except:
        pass
print("Old files deleted")

# Regenerate with fixed amounts
print(f"\nRegenerating {BATCH_SIZE * NUM_BATCHES:,} transactions...")
total_written = 0
total_flagged = 0

for batch_num in range(2, NUM_BATCHES + 2):
    transactions  = [make_transaction(i, batch_num, time_offset_hours=batch_num-2) for i in range(1, BATCH_SIZE + 1)]
    batch_flagged = sum(1 for t in transactions if t["transaction_type"] != "normal")
    total_flagged += batch_flagged
    total_written += len(transactions)

    file_path = f"{VOLUME_PATH}/batch_{str(batch_num).zfill(3)}.json"
    with open(file_path, "w") as f:
        for txn in transactions:
            f.write(json.dumps(txn) + "\n")

    print(f"Batch {batch_num-1:>2}/10 done -> {file_path.split('/')[-1]} ({batch_flagged} suspicious)")

print(f"\nTotal written    : {total_written:,}")
print(f"Total suspicious : {total_flagged:,}")
print(f"Suspicious rate  : {round(total_flagged/total_written*100,1)}%")

Deleting old batch files...
Old files deleted

Regenerating 100,000 transactions...
Batch  1/10 done -> batch_002.json (500 suspicious)
Batch  2/10 done -> batch_003.json (500 suspicious)
Batch  3/10 done -> batch_004.json (500 suspicious)
Batch  4/10 done -> batch_005.json (500 suspicious)
Batch  5/10 done -> batch_006.json (500 suspicious)
Batch  6/10 done -> batch_007.json (500 suspicious)
Batch  7/10 done -> batch_008.json (500 suspicious)
Batch  8/10 done -> batch_009.json (500 suspicious)
Batch  9/10 done -> batch_010.json (500 suspicious)
Batch 10/10 done -> batch_011.json (500 suspicious)

Total written    : 100,000
Total suspicious : 5,000
Suspicious rate  : 5.0%


In [0]:
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.bronze_transactions")
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.silver_transactions")
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.gold_sar_reports")
dbutils.fs.rm("/Volumes/aml_pipeline/transactions/raw_data/_checkpoints", recurse=True)
print("All clear — ready to rerun pipeline")


All clear — ready to rerun pipeline


In [0]:
import requests
from pyspark.sql.functions import col, udf, when, lit, current_timestamp, concat, date_format
from pyspark.sql.types import DoubleType, StringType, StructType, StructField, IntegerType

RAW_PATH        = "/Volumes/aml_pipeline/transactions/raw_data/"
BRONZE_TABLE    = "aml_pipeline.transactions.bronze_transactions"
SILVER_TABLE    = "aml_pipeline.transactions.silver_transactions"
GOLD_TABLE      = "aml_pipeline.transactions.gold_sar_reports"
CHECKPOINT_PATH = "/Volumes/aml_pipeline/transactions/raw_data/_checkpoints/bronze_v4"

# STEP 1: Bronze
print("STEP 1: Bronze ingestion...")

schema = StructType([
    StructField("transaction_id",   StringType(),  True),
    StructField("timestamp",        StringType(),  True),
    StructField("message_type",     StringType(),  True),
    StructField("batch_number",     IntegerType(), True),
    StructField("sender_name",      StringType(),  True),
    StructField("sender_account",   StringType(),  True),
    StructField("sender_country",   StringType(),  True),
    StructField("sender_address",   StringType(),  True),
    StructField("receiver_name",    StringType(),  True),
    StructField("receiver_account", StringType(),  True),
    StructField("receiver_country", StringType(),  True),
    StructField("amount",           DoubleType(),  True),
    StructField("currency",         StringType(),  True),
    StructField("purpose_code",     StringType(),  True),
    StructField("transaction_type", StringType(),  True),
])

bronze_query = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
         .schema(schema)
         .load(RAW_PATH)
         .withColumn("ingestion_timestamp", current_timestamp())
         .withColumn("source_file",         col("_metadata.file_path"))
         .withColumn("pipeline_layer",      lit("bronze"))
         .writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", CHECKPOINT_PATH)
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(BRONZE_TABLE)
)
bronze_query.awaitTermination()

bronze_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze complete: {bronze_count:,} records")
assert bronze_count == 100_000, f"ERROR: Expected 100,000 but got {bronze_count:,}. Stop and fix before continuing."

# STEP 2: Silver
print("\nSTEP 2: Silver enrichment...")

try:
    rates = requests.get("https://api.frankfurter.app/latest?from=USD", timeout=10).json().get("rates", {})
    rates["USD"] = 1.0
    print(f"Live exchange rates fetched ({len(rates)} currencies)")
except:
    rates = {"USD":1.0,"EUR":0.92,"GBP":0.79,"JPY":149.5,"CHF":0.89,"CAD":1.36,"AUD":1.53,"SGD":1.34}
    print("Using fallback exchange rates")

try:
    lines = requests.get("https://www.treasury.gov/ofac/downloads/sdn.csv", timeout=15).text.split("\n")
    OFAC  = set()
    for line in lines[:500]:
        parts = line.split(",")
        if len(parts) > 1:
            n = parts[1].strip().strip('"').lower()
            if n:
                OFAC.add(n)
    print(f"OFAC list loaded ({len(OFAC)} sanctioned names)")
except:
    OFAC = {"viktor bout","semion mogilevich","ramzan kadyrov","ali khamenei","kim jong un"}
    print("Using fallback OFAC list")

HIGH_RISK           = {"KP","IR","MM","RU","BY","CU","SY","YE"}
LARGE_TXN_THRESHOLD = 500_000

def to_usd(amount, currency):
    if not amount or not currency:
        return None
    return round(float(amount) / float(rates.get(currency, 1.0)), 2)

def check_ofac(name):
    if not name:
        return "CLEAN"
    n = name.lower().strip()
    if n in OFAC:
        return "SANCTIONS_HIT"
    for s in OFAC:
        if s and len(s) > 5 and (s in n or n in s):
            return "SANCTIONS_HIT"
    return "CLEAN"

def check_travel_rule(address, sender, s_acct, receiver, r_acct):
    missing = [f for f, v in [
        ("sender_address",   address),
        ("sender_name",      sender),
        ("sender_account",   s_acct),
        ("receiver_name",    receiver),
        ("receiver_account", r_acct)
    ] if not v or not v.strip()]
    return f"TRAVEL_RULE_VIOLATION: missing {', '.join(missing)}" if missing else "COMPLIANT"

udf_usd  = udf(to_usd, DoubleType())
udf_ofac = udf(check_ofac, StringType())
udf_tr   = udf(check_travel_rule, StringType())

silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("amount_usd",                udf_usd(col("amount"), col("currency")))
    .withColumn("sender_sanctions_status",   udf_ofac(col("sender_name")))
    .withColumn("receiver_sanctions_status", udf_ofac(col("receiver_name")))
    .withColumn("travel_rule_status",        udf_tr(col("sender_address"), col("sender_name"),
                                             col("sender_account"), col("receiver_name"),
                                             col("receiver_account")))
    .withColumn("high_risk_country",         col("sender_country").isin(list(HIGH_RISK)))
    .withColumn("large_transaction",         col("amount_usd") > LARGE_TXN_THRESHOLD)
    .withColumn("is_flagged",
        (col("sender_sanctions_status") == "SANCTIONS_HIT") |
        (col("receiver_sanctions_status") == "SANCTIONS_HIT") |
        (col("travel_rule_status") != "COMPLIANT") |
        (col("high_risk_country") == True) |
        (col("large_transaction") == True))
    .withColumn("flag_reason",
        when(col("sender_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned sender"))
        .when(col("receiver_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned receiver"))
        .when(col("travel_rule_status") != "COMPLIANT", col("travel_rule_status"))
        .when(col("high_risk_country") == True, lit("HIGH_RISK_COUNTRY"))
        .when(col("large_transaction") == True, lit("LARGE_TRANSACTION_>500K_USD"))
        .otherwise(lit("NONE")))
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("pipeline_layer",   lit("silver"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

silver_count  = spark.table(SILVER_TABLE).count()
flagged_count = spark.table(SILVER_TABLE).filter("is_flagged = true").count()
clean_count   = silver_count - flagged_count
print(f"Silver complete: {silver_count:,} records")
print(f"  Flagged : {flagged_count:,} ({round(flagged_count/silver_count*100,1)}%)")
print(f"  Clean   : {clean_count:,} ({round(clean_count/silver_count*100,1)}%)")

# STEP 3: Gold
print("\nSTEP 3: Gold SAR reports...")

gold_df = (
    spark.table(SILVER_TABLE)
    .filter(col("is_flagged") == True)
    .withColumn("sar_severity",
        when((col("sender_sanctions_status") == "SANCTIONS_HIT") |
             (col("receiver_sanctions_status") == "SANCTIONS_HIT"), lit("CRITICAL"))
        .when(col("high_risk_country") == True, lit("HIGH"))
        .when(col("travel_rule_status") != "COMPLIANT", lit("HIGH"))
        .otherwise(lit("MEDIUM")))
    .withColumn("sar_reference",
        concat(lit("SAR-"), date_format(current_timestamp(), "yyyyMMdd"),
               lit("-"), col("transaction_id").substr(1, 8)))
    .withColumn("report_status",         lit("PENDING_REVIEW"))
    .withColumn("reporting_institution", lit("SentinelFlow Demo Bank"))
    .withColumn("filing_deadline",       date_format(current_timestamp(), "yyyy-MM-dd"))
    .withColumn("gold_timestamp",        current_timestamp())
    .withColumn("pipeline_layer",        lit("gold"))
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)

gold_count = spark.table(GOLD_TABLE).count()
critical   = spark.table(GOLD_TABLE).filter("sar_severity = 'CRITICAL'").count()
high       = spark.table(GOLD_TABLE).filter("sar_severity = 'HIGH'").count()
medium     = spark.table(GOLD_TABLE).filter("sar_severity = 'MEDIUM'").count()

print(f"\n{'=' * 55}")
print(f"  SENTINELFLOW PIPELINE COMPLETE")
print(f"{'=' * 55}")
print(f"  Bronze    : {bronze_count:,} raw transactions")
print(f"  Silver    : {silver_count:,} screened transactions")
print(f"  Gold      : {gold_count:,} SAR reports")
print(f"  Flag rate : {round(gold_count/silver_count*100,1)}%")
print(f"\n  SAR Severity:")
print(f"  CRITICAL  : {critical:,}")
print(f"  HIGH      : {high:,}")
print(f"  MEDIUM    : {medium:,}")
print(f"\n  Flag reason breakdown:")
spark.table(GOLD_TABLE).groupBy("flag_reason").count().orderBy("count", ascending=False).show(truncate=False)

# ── STEP 1: Bronze ───────────────────────────────────────────
print("=" * 55)
print("STEP 1: Bronze — ingesting 100,000 transactions...")
print("=" * 55)

schema = StructType([
    StructField("transaction_id",   StringType(),  True),
    StructField("timestamp",        StringType(),  True),
    StructField("message_type",     StringType(),  True),
    StructField("batch_number",     IntegerType(), True),
    StructField("sender_name",      StringType(),  True),
    StructField("sender_account",   StringType(),  True),
    StructField("sender_country",   StringType(),  True),
    StructField("sender_address",   StringType(),  True),
    StructField("receiver_name",    StringType(),  True),
    StructField("receiver_account", StringType(),  True),
    StructField("receiver_country", StringType(),  True),
    StructField("amount",           DoubleType(),  True),
    StructField("currency",         StringType(),  True),
    StructField("purpose_code",     StringType(),  True),
    StructField("transaction_type", StringType(),  True),
])

bronze_query = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
         .schema(schema)
         .load(RAW_PATH)
         .withColumn("ingestion_timestamp", current_timestamp())
         .withColumn("source_file",         col("_metadata.file_path"))
         .withColumn("pipeline_layer",      lit("bronze"))
         .writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", CHECKPOINT_PATH)
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(BRONZE_TABLE)
)
bronze_query.awaitTermination()

bronze_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze complete: {bronze_count:,} records")

# ── STEP 2: Silver ───────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 2: Silver — enrichment + compliance screening...")
print("=" * 55)

# Fetch live exchange rates
try:
    rates = requests.get("https://api.frankfurter.app/latest?from=USD", timeout=10).json().get("rates", {})
    rates["USD"] = 1.0
    print(f"Live exchange rates fetched ({len(rates)} currencies)")
except:
    rates = {"USD":1.0,"EUR":0.92,"GBP":0.79,"JPY":149.5,"CHF":0.89,"CAD":1.36,"AUD":1.53,"SGD":1.34}
    print("Using fallback exchange rates")

# Fetch OFAC sanctions list
try:
    lines = requests.get("https://www.treasury.gov/ofac/downloads/sdn.csv", timeout=15).text.split("\n")
    OFAC  = set()
    for line in lines[:500]:
        parts = line.split(",")
        if len(parts) > 1:
            n = parts[1].strip().strip('"').lower()
            if n:
                OFAC.add(n)
    print(f"OFAC list loaded ({len(OFAC)} sanctioned names)")
except:
    OFAC = {"viktor bout","semion mogilevich","ramzan kadyrov","ali khamenei","kim jong un"}
    print("Using fallback OFAC list")

HIGH_RISK = {"KP","IR","MM","RU","BY","CU","SY","YE"}

# ── Key fix: threshold raised to $500,000 ───────────────────
# Real banks flag transactions over $500k as large.
# $50k was catching normal business payments — too noisy.
LARGE_TXN_THRESHOLD = 500_000

def to_usd(amount, currency):
    if not amount or not currency:
        return None
    return round(float(amount) / float(rates.get(currency, 1.0)), 2)

def check_ofac(name):
    if not name:
        return "CLEAN"
    n = name.lower().strip()
    if n in OFAC:
        return "SANCTIONS_HIT"
    for s in OFAC:
        if s and len(s) > 5 and (s in n or n in s):
            return "SANCTIONS_HIT"
    return "CLEAN"

def check_travel_rule(address, sender, s_acct, receiver, r_acct):
    missing = [f for f, v in [
        ("sender_address",  address),
        ("sender_name",     sender),
        ("sender_account",  s_acct),
        ("receiver_name",   receiver),
        ("receiver_account",r_acct)
    ] if not v or not v.strip()]
    return f"TRAVEL_RULE_VIOLATION: missing {', '.join(missing)}" if missing else "COMPLIANT"

udf_usd  = udf(to_usd, DoubleType())
udf_ofac = udf(check_ofac, StringType())
udf_tr   = udf(check_travel_rule, StringType())

silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("amount_usd",                udf_usd(col("amount"), col("currency")))
    .withColumn("sender_sanctions_status",   udf_ofac(col("sender_name")))
    .withColumn("receiver_sanctions_status", udf_ofac(col("receiver_name")))
    .withColumn("travel_rule_status",        udf_tr(col("sender_address"), col("sender_name"),
                                             col("sender_account"), col("receiver_name"),
                                             col("receiver_account")))
    .withColumn("high_risk_country",         col("sender_country").isin(list(HIGH_RISK)))
    .withColumn("large_transaction",         col("amount_usd") > LARGE_TXN_THRESHOLD)
    .withColumn("is_flagged",
        (col("sender_sanctions_status") == "SANCTIONS_HIT") |
        (col("receiver_sanctions_status") == "SANCTIONS_HIT") |
        (col("travel_rule_status") != "COMPLIANT") |
        (col("high_risk_country") == True) |
        (col("large_transaction") == True))
    .withColumn("flag_reason",
        when(col("sender_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned sender"))
        .when(col("receiver_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned receiver"))
        .when(col("travel_rule_status") != "COMPLIANT", col("travel_rule_status"))
        .when(col("high_risk_country") == True, lit("HIGH_RISK_COUNTRY"))
        .when(col("large_transaction") == True, lit("LARGE_TRANSACTION_>500K_USD"))
        .otherwise(lit("NONE")))
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("pipeline_layer",   lit("silver"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

silver_count  = spark.table(SILVER_TABLE).count()
flagged_count = spark.table(SILVER_TABLE).filter("is_flagged = true").count()
clean_count   = silver_count - flagged_count
print(f"Silver complete: {silver_count:,} records")
print(f"   Flagged : {flagged_count:,} ({round(flagged_count/silver_count*100,1)}%)")
print(f"   Clean   : {clean_count:,} ({round(clean_count/silver_count*100,1)}%)")

# ── STEP 3: Gold ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 3: Gold — generating SAR reports...")
print("=" * 55)

gold_df = (
    spark.table(SILVER_TABLE)
    .filter(col("is_flagged") == True)
    .withColumn("sar_severity",
        when((col("sender_sanctions_status") == "SANCTIONS_HIT") |
             (col("receiver_sanctions_status") == "SANCTIONS_HIT"), lit("CRITICAL"))
        .when(col("high_risk_country") == True, lit("HIGH"))
        .when(col("travel_rule_status") != "COMPLIANT", lit("HIGH"))
        .otherwise(lit("MEDIUM")))
    .withColumn("sar_reference",
        concat(lit("SAR-"), date_format(current_timestamp(), "yyyyMMdd"),
               lit("-"), col("transaction_id").substr(1, 8)))
    .withColumn("report_status",         lit("PENDING_REVIEW"))
    .withColumn("reporting_institution", lit("SentinelFlow Demo Bank"))
    .withColumn("filing_deadline",       date_format(current_timestamp(), "yyyy-MM-dd"))
    .withColumn("gold_timestamp",        current_timestamp())
    .withColumn("pipeline_layer",        lit("gold"))
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)

gold_count    = spark.table(GOLD_TABLE).count()
critical      = spark.table(GOLD_TABLE).filter("sar_severity = 'CRITICAL'").count()
high          = spark.table(GOLD_TABLE).filter("sar_severity = 'HIGH'").count()
medium        = spark.table(GOLD_TABLE).filter("sar_severity = 'MEDIUM'").count()

# ── FINAL SUMMARY ─────────────────────────────────────────────
print(f"\n{'=' * 55}")
print(f"  SENTINELFLOW — PIPELINE COMPLETE")
print(f"{'=' * 55}")
print(f"  🥉 Bronze : {bronze_count:,} raw transactions")
print(f"  🥈 Silver : {silver_count:,} screened transactions")
print(f"  🥇 Gold   : {gold_count:,} SAR reports")
print(f"  Flag rate : {round(gold_count/silver_count*100,1)}%")
print(f"\n  SAR Severity:")
print(f"  🔴 CRITICAL : {critical:,}")
print(f"  🟠 HIGH     : {high:,}")
print(f"  🟡 MEDIUM   : {medium:,}")
print(f"\n  Flag reason breakdown:")
spark.table(GOLD_TABLE).groupBy("flag_reason").count().orderBy("count", ascending=False).show(truncate=False)

STEP 1: Bronze ingestion...
Bronze complete: 100,000 records

STEP 2: Silver enrichment...
Live exchange rates fetched (30 currencies)
OFAC list loaded (470 sanctioned names)
Silver complete: 100,000 records
  Flagged : 12,884 (12.9%)
  Clean   : 87,116 (87.1%)

STEP 3: Gold SAR reports...

  SENTINELFLOW PIPELINE COMPLETE
  Bronze    : 100,000 raw transactions
  Silver    : 100,000 screened transactions
  Gold      : 12,884 SAR reports
  Flag rate : 12.9%

  SAR Severity:
  CRITICAL  : 9,867
  HIGH      : 3,017
  MEDIUM    : 0

  Flag reason breakdown:
+---------------------------------------------+-----+
|flag_reason                                  |count|
+---------------------------------------------+-----+
|OFAC: Sanctioned sender                      |5162 |
|OFAC: Sanctioned receiver                    |4705 |
|TRAVEL_RULE_VIOLATION: missing sender_address|1522 |
|HIGH_RISK_COUNTRY                            |1495 |
+---------------------------------------------+-----+

STEP 1:

In [0]:
# Check what normal names are being falsely matched
false_hits = spark.table("aml_pipeline.transactions.silver_transactions") \
    .filter("sender_sanctions_status = 'SANCTIONS_HIT'") \
    .filter("transaction_type = 'normal'") \
    .select("sender_name", "sender_sanctions_status", "transaction_type") \
    .distinct() \
    .show(20, truncate=False)
    

+------------------+-----------------------+----------------+
|sender_name       |sender_sanctions_status|transaction_type|
+------------------+-----------------------+----------------+
|Mohammed Al-Rashid|SANCTIONS_HIT          |normal          |
+------------------+-----------------------+----------------+



In [0]:
# Drop tables and checkpoints for clean rerun
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.bronze_transactions")
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.silver_transactions")
spark.sql("DROP TABLE IF EXISTS aml_pipeline.transactions.gold_sar_reports")
dbutils.fs.rm("/Volumes/aml_pipeline/transactions/raw_data/_checkpoints", recurse=True)
print("All clear")

All clear


In [0]:
import requests
from pyspark.sql.functions import col, udf, when, lit, current_timestamp, concat, date_format
from pyspark.sql.types import DoubleType, StringType, StructType, StructField, IntegerType

RAW_PATH        = "/Volumes/aml_pipeline/transactions/raw_data/"
BRONZE_TABLE    = "aml_pipeline.transactions.bronze_transactions"
SILVER_TABLE    = "aml_pipeline.transactions.silver_transactions"
GOLD_TABLE      = "aml_pipeline.transactions.gold_sar_reports"
CHECKPOINT_PATH = "/Volumes/aml_pipeline/transactions/raw_data/_checkpoints/bronze_v5"

# STEP 1: Bronze
print("STEP 1: Bronze ingestion...")

schema = StructType([
    StructField("transaction_id",   StringType(),  True),
    StructField("timestamp",        StringType(),  True),
    StructField("message_type",     StringType(),  True),
    StructField("batch_number",     IntegerType(), True),
    StructField("sender_name",      StringType(),  True),
    StructField("sender_account",   StringType(),  True),
    StructField("sender_country",   StringType(),  True),
    StructField("sender_address",   StringType(),  True),
    StructField("receiver_name",    StringType(),  True),
    StructField("receiver_account", StringType(),  True),
    StructField("receiver_country", StringType(),  True),
    StructField("amount",           DoubleType(),  True),
    StructField("currency",         StringType(),  True),
    StructField("purpose_code",     StringType(),  True),
    StructField("transaction_type", StringType(),  True),
])

bronze_query = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
         .schema(schema)
         .load(RAW_PATH)
         .withColumn("ingestion_timestamp", current_timestamp())
         .withColumn("source_file",         col("_metadata.file_path"))
         .withColumn("pipeline_layer",      lit("bronze"))
         .writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", CHECKPOINT_PATH)
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(BRONZE_TABLE)
)
bronze_query.awaitTermination()

bronze_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze complete: {bronze_count:,} records")
assert bronze_count == 100_000, f"ERROR: Expected 100,000 but got {bronze_count:,}"

# STEP 2: Silver
print("\nSTEP 2: Silver enrichment...")

try:
    rates = requests.get("https://api.frankfurter.app/latest?from=USD", timeout=10).json().get("rates", {})
    rates["USD"] = 1.0
    print(f"Live exchange rates fetched ({len(rates)} currencies)")
except:
    rates = {"USD":1.0,"EUR":0.92,"GBP":0.79,"JPY":149.5,"CHF":0.89,"CAD":1.36,"AUD":1.53,"SGD":1.34}
    print("Using fallback exchange rates")

try:
    lines = requests.get("https://www.treasury.gov/ofac/downloads/sdn.csv", timeout=15).text.split("\n")
    OFAC  = set()
    for line in lines[:500]:
        parts = line.split(",")
        if len(parts) > 1:
            n = parts[1].strip().strip('"').lower()
            if n:
                OFAC.add(n)
    print(f"OFAC list loaded ({len(OFAC)} sanctioned names)")
except:
    OFAC = {"viktor bout","semion mogilevich","ramzan kadyrov","ali khamenei","kim jong un"}
    print("Using fallback OFAC list")

HIGH_RISK           = {"KP","IR","MM","RU","BY","CU","SY","YE"}
LARGE_TXN_THRESHOLD = 500_000

def to_usd(amount, currency):
    if not amount or not currency:
        return None
    return round(float(amount) / float(rates.get(currency, 1.0)), 2)

# KEY FIX: exact matching only — no partial/substring matching
# This mirrors how real OFAC screening works at the name level
# before a human reviews potential fuzzy matches
def check_ofac(name):
    if not name:
        return "CLEAN"
    n = name.lower().strip()
    # Only flag if the FULL name is an exact match
    if n in OFAC:
        return "SANCTIONS_HIT"
    return "CLEAN"

def check_travel_rule(address, sender, s_acct, receiver, r_acct):
    missing = [f for f, v in [
        ("sender_address",   address),
        ("sender_name",      sender),
        ("sender_account",   s_acct),
        ("receiver_name",    receiver),
        ("receiver_account", r_acct)
    ] if not v or not v.strip()]
    return f"TRAVEL_RULE_VIOLATION: missing {', '.join(missing)}" if missing else "COMPLIANT"

udf_usd  = udf(to_usd, DoubleType())
udf_ofac = udf(check_ofac, StringType())
udf_tr   = udf(check_travel_rule, StringType())

silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("amount_usd",                udf_usd(col("amount"), col("currency")))
    .withColumn("sender_sanctions_status",   udf_ofac(col("sender_name")))
    .withColumn("receiver_sanctions_status", udf_ofac(col("receiver_name")))
    .withColumn("travel_rule_status",        udf_tr(col("sender_address"), col("sender_name"),
                                             col("sender_account"), col("receiver_name"),
                                             col("receiver_account")))
    .withColumn("high_risk_country",         col("sender_country").isin(list(HIGH_RISK)))
    .withColumn("large_transaction",         col("amount_usd") > LARGE_TXN_THRESHOLD)
    .withColumn("is_flagged",
        (col("sender_sanctions_status") == "SANCTIONS_HIT") |
        (col("receiver_sanctions_status") == "SANCTIONS_HIT") |
        (col("travel_rule_status") != "COMPLIANT") |
        (col("high_risk_country") == True) |
        (col("large_transaction") == True))
    .withColumn("flag_reason",
        when(col("sender_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned sender"))
        .when(col("receiver_sanctions_status") == "SANCTIONS_HIT", lit("OFAC: Sanctioned receiver"))
        .when(col("travel_rule_status") != "COMPLIANT", col("travel_rule_status"))
        .when(col("high_risk_country") == True, lit("HIGH_RISK_COUNTRY"))
        .when(col("large_transaction") == True, lit("LARGE_TRANSACTION_>500K_USD"))
        .otherwise(lit("NONE")))
    .withColumn("silver_timestamp", current_timestamp())
    .withColumn("pipeline_layer",   lit("silver"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

silver_count  = spark.table(SILVER_TABLE).count()
flagged_count = spark.table(SILVER_TABLE).filter("is_flagged = true").count()
clean_count   = silver_count - flagged_count
print(f"Silver complete: {silver_count:,} records")
print(f"  Flagged : {flagged_count:,} ({round(flagged_count/silver_count*100,1)}%)")
print(f"  Clean   : {clean_count:,} ({round(clean_count/silver_count*100,1)}%)")

# STEP 3: Gold
print("\nSTEP 3: Gold SAR reports...")

gold_df = (
    spark.table(SILVER_TABLE)
    .filter(col("is_flagged") == True)
    .withColumn("sar_severity",
        when((col("sender_sanctions_status") == "SANCTIONS_HIT") |
             (col("receiver_sanctions_status") == "SANCTIONS_HIT"), lit("CRITICAL"))
        .when(col("high_risk_country") == True, lit("HIGH"))
        .when(col("travel_rule_status") != "COMPLIANT", lit("HIGH"))
        .otherwise(lit("MEDIUM")))
    .withColumn("sar_reference",
        concat(lit("SAR-"), date_format(current_timestamp(), "yyyyMMdd"),
               lit("-"), col("transaction_id").substr(1, 8)))
    .withColumn("report_status",         lit("PENDING_REVIEW"))
    .withColumn("reporting_institution", lit("SentinelFlow Demo Bank"))
    .withColumn("filing_deadline",       date_format(current_timestamp(), "yyyy-MM-dd"))
    .withColumn("gold_timestamp",        current_timestamp())
    .withColumn("pipeline_layer",        lit("gold"))
)

gold_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)

gold_count = spark.table(GOLD_TABLE).count()
critical   = spark.table(GOLD_TABLE).filter("sar_severity = 'CRITICAL'").count()
high       = spark.table(GOLD_TABLE).filter("sar_severity = 'HIGH'").count()
medium     = spark.table(GOLD_TABLE).filter("sar_severity = 'MEDIUM'").count()

print(f"\n{'=' * 55}")
print(f"  SENTINELFLOW PIPELINE COMPLETE")
print(f"{'=' * 55}")
print(f"  Bronze    : {bronze_count:,} raw transactions")
print(f"  Silver    : {silver_count:,} screened transactions")
print(f"  Gold      : {gold_count:,} SAR reports")
print(f"  Flag rate : {round(gold_count/silver_count*100,1)}%")
print(f"\n  SAR Severity:")
print(f"  CRITICAL  : {critical:,}")
print(f"  HIGH      : {high:,}")
print(f"  MEDIUM    : {medium:,}")
print(f"\n  Flag reason breakdown:")
spark.table(GOLD_TABLE).groupBy("flag_reason").count().orderBy("count", ascending=False).show(truncate=False)

STEP 1: Bronze ingestion...
Bronze complete: 100,000 records

STEP 2: Silver enrichment...
Live exchange rates fetched (30 currencies)
OFAC list loaded (470 sanctioned names)
Silver complete: 100,000 records
  Flagged : 3,356 (3.4%)
  Clean   : 96,644 (96.6%)

STEP 3: Gold SAR reports...

  SENTINELFLOW PIPELINE COMPLETE
  Bronze    : 100,000 raw transactions
  Silver    : 100,000 screened transactions
  Gold      : 3,356 SAR reports
  Flag rate : 3.4%

  SAR Severity:
  CRITICAL  : 0
  HIGH      : 3,356
  MEDIUM    : 0

  Flag reason breakdown:
+---------------------------------------------+-----+
|flag_reason                                  |count|
+---------------------------------------------+-----+
|TRAVEL_RULE_VIOLATION: missing sender_address|1703 |
|HIGH_RISK_COUNTRY                            |1653 |
+---------------------------------------------+-----+

